In [ ]:
import os
os.environ["TABPFN_TOKEN"] = "tabpfn_sk_HaipTnzMXWTAGKvzk656n8PmST5zUun9I03TPSx_51Q "

In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score, matthews_corrcoef,
    confusion_matrix
)
from tabpfn import TabPFNClassifier

RANDOM_STATE = 42
LEAKAGE_PRONE_FEATURES = ["PageValues", "BounceRates", "ExitRates"]

In [7]:
df = pd.read_csv("online_shoppers_intention.csv")

NOMINAL_INT_COLS = ["OperatingSystems", "Browser", "Region", "TrafficType"]
df[NOMINAL_INT_COLS] = df[NOMINAL_INT_COLS].astype("category")
df_encoded = pd.get_dummies(df, drop_first=True, dtype=int)

y = df_encoded["Revenue"]
X_main = df_encoded.drop("Revenue", axis=1)
X_early = X_main.drop(columns=LEAKAGE_PRONE_FEATURES)

In [8]:
def fit_and_score_tabpfn(X, y, label, max_train_rows=None):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
    )

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)

    # Optional: subsample if you hit memory/speed limits (uncomment to use)
    # if max_train_rows and len(X_train_s) > max_train_rows:
    #     rng = np.random.RandomState(RANDOM_STATE)
    #     idx = rng.choice(len(X_train_s), max_train_rows, replace=False)
    #     X_train_s, y_train = X_train_s[idx], y_train.iloc[idx]

    model = TabPFNClassifier(
        device="cpu",                     # change to "cuda" if you have a GPU
        ignore_pretraining_limits=True,   # allow slightly-over-spec row/feature counts
        random_state=RANDOM_STATE,
    )
    model.fit(X_train_s, y_train)

    y_prob = model.predict_proba(X_test_s)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)

    row = {
        "Model": "TabPFN",
        "Feature Set": label,
        "Accuracy": accuracy_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob),
        "PR-AUC": average_precision_score(y_test, y_prob),
        "Purchase Precision": precision_score(y_test, y_pred),
        "Purchase Recall": recall_score(y_test, y_pred),
        "Purchase F1": f1_score(y_test, y_pred),
        "MCC": matthews_corrcoef(y_test, y_pred),
    }
    return row, model, X_test_s, y_test, y_prob

In [9]:
tabpfn_full_row, tabpfn_full_model, X_test_s_full, y_test_full, y_prob_full = fit_and_score_tabpfn(
    X_main, y, "Full feature set (with PageValues/BounceRates/ExitRates)"
)

tabpfn_early_row, tabpfn_early_model, X_test_s_early, y_test_early, y_prob_early = fit_and_score_tabpfn(
    X_early, y, "Leakage-free feature set"
)

tabpfn_results_df = pd.DataFrame([tabpfn_full_row, tabpfn_early_row])
tabpfn_results_df

,Model,Feature Set,Accuracy,ROC-AUC,PR-AUC,Purchase Precision,Purchase Recall,Purchase F1,MCC
0,TabPFN,Full feature set (with PageValues/BounceRates/...,0.903893,0.935719,0.762620,0.736156,0.591623,0.656023,0.605784
1,TabPFN,Leakage-free feature set,0.847121,0.780543,0.382783,0.567568,0.054974,0.100239,0.140764


In [13]:
leakage_results = pd.read_csv("leakage_ablation_results_with_xgb.csv")

tabpfn_for_table2 = pd.DataFrame([
    {"Feature set": tabpfn_full_row["Feature Set"], "Model": "TabPFN",
     "Accuracy": tabpfn_full_row["Accuracy"], "ROC-AUC": tabpfn_full_row["ROC-AUC"],
     "PR-AUC": tabpfn_full_row["PR-AUC"],
     "F1": tabpfn_full_row["Purchase F1"], "MCC": tabpfn_full_row["MCC"]},
    {"Feature set": tabpfn_early_row["Feature Set"], "Model": "TabPFN",
     "Accuracy": tabpfn_early_row["Accuracy"], "ROC-AUC": tabpfn_early_row["ROC-AUC"],
     "PR-AUC": tabpfn_early_row["PR-AUC"],
     "F1": tabpfn_early_row["Purchase F1"], "MCC": tabpfn_early_row["MCC"]},
])

leakage_results_with_tabpfn = pd.concat([leakage_results, tabpfn_for_table2], ignore_index=True)
leakage_results_with_tabpfn.to_csv("leakage_ablation_results_with_tabpfn.csv", index=False)
leakage_results_with_tabpfn

,Feature set,Model,Accuracy,ROC-AUC,PR-AUC,F1,MCC
0,Full feature set (with PageValues/BounceRates/...,Logistic Regression,0.836600,0.891800,0.620800,0.583200,0.504200
1,Full feature set (with PageValues/BounceRates/...,Random Forest,0.886100,0.924100,0.722100,0.660200,0.594700
2,Early-prediction (excluding PageValues/BounceR...,Logistic Regression,0.656500,0.716400,0.308800,0.374900,0.236800
3,Early-prediction (excluding PageValues/BounceR...,Random Forest,0.797600,0.757800,0.351500,0.381700,0.261600
4,Full feature set (with PageValues/BounceRates/...,XGBoost,0.882000,0.922500,0.716400,0.654000,0.587100
5,Early-prediction (excluding PageValues/BounceR...,XGBoost,0.757500,0.732500,0.319500,0.369200,0.232700
6,Full feature set (with PageValues/BounceRates/...,TabPFN,0.903893,0.935719,0.762620,0.656023,0.605784
7,Leakage-free feature set,TabPFN,0.847121,0.780543,0.382783,0.100239,0.140764


In [15]:
leakage_results = pd.read_csv("leakage_ablation_results_with_xgb.csv")

tabpfn_for_table2 = pd.DataFrame([
    {"Model": "TabPFN", "Feature Set": tabpfn_full_row["Feature Set"],
     "Accuracy": tabpfn_full_row["Accuracy"], "ROC-AUC": tabpfn_full_row["ROC-AUC"],
     "PR-AUC": tabpfn_full_row["PR-AUC"]},
    {"Model": "TabPFN", "Feature Set": tabpfn_early_row["Feature Set"],
     "Accuracy": tabpfn_early_row["Accuracy"], "ROC-AUC": tabpfn_early_row["ROC-AUC"],
     "PR-AUC": tabpfn_early_row["PR-AUC"]},
])

leakage_results_with_tabpfn = pd.concat([leakage_results, tabpfn_for_table2], ignore_index=True)
leakage_results_with_tabpfn.to_csv("leakage_ablation_results_with_tabpfn.csv", index=False)
leakage_results_with_tabpfn

,Feature set,Model,Accuracy,ROC-AUC,PR-AUC,F1,MCC,Feature Set
0,Full feature set (with PageValues/BounceRates/...,Logistic Regression,0.836600,0.891800,0.620800,0.5832,0.5042,NaN
1,Full feature set (with PageValues/BounceRates/...,Random Forest,0.886100,0.924100,0.722100,0.6602,0.5947,NaN
2,Early-prediction (excluding PageValues/BounceR...,Logistic Regression,0.656500,0.716400,0.308800,0.3749,0.2368,NaN
3,Early-prediction (excluding PageValues/BounceR...,Random Forest,0.797600,0.757800,0.351500,0.3817,0.2616,NaN
4,Full feature set (with PageValues/BounceRates/...,XGBoost,0.882000,0.922500,0.716400,0.6540,0.5871,NaN
5,Early-prediction (excluding PageValues/BounceR...,XGBoost,0.757500,0.732500,0.319500,0.3692,0.2327,NaN
6,NaN,TabPFN,0.903893,0.935719,0.762620,NaN,NaN,Full feature set (with PageValues/BounceRates/...
7,NaN,TabPFN,0.847121,0.780543,0.382783,NaN,NaN,Leakage-free feature set


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, matthews_corrcoef

y_prob = tabpfn_full_model.predict_proba(X_test_s_full)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

print(f"Purchase Precision : {precision_score(y_test_full, y_pred)*100:.2f}%")
print(f"Purchase Recall    : {recall_score(y_test_full, y_pred)*100:.2f}%")
print(f"Purchase F1 Score  : {f1_score(y_test_full, y_pred):.4f}")
print(f"MCC                : {matthews_corrcoef(y_test_full, y_pred):.4f}")
